In [1]:
import os
from pathlib import Path
REPO = Path.cwd().resolve().parents[1]          # docs/notebooks -> repo root
GOLDEN = Path(os.environ.get("CAL_DISP_GOLDEN_DIR", REPO / "test_golden"))
SCRATCH = Path(os.environ.get("TMPDIR", REPO / "tmp")) / "cal_disp_notebooks"; SCRATCH.mkdir(parents=True, exist_ok=True)

# 13 — Download robustness: timeouts, atomic writes, body checks

**Fix.** The staging helpers (`cal_disp.download`) no longer hang, no longer keep corrupt files,
and no longer report success when nothing was downloaded.

**Where the defect was** (paths at commit `d0901b3`, the state before the fix):

| Defect | Location |
|---|---|
| `session.get(url)` without `timeout`: a stalled server blocks the stage forever | `src/cal_disp/download/_stage_unr.py:158`, `:297` |
| `output_path.write_bytes(response.content)`: whatever the server sent becomes the file, written non-atomically | `_stage_unr.py:161`, `:300` |
| `if output_path.exists(): return`: a partial file or a saved HTML error page counts as complete forever | `_stage_unr.py:148`, `:290` |
| retries only on 502/503/504, not on 429 (rate limit) or 500 | `_stage_unr.py:86` |
| zero TROPO scenes found: downloads nothing, returns, CLI prints "Download complete" | `_stage_tropo.py:123-129`, `cli/download.py:225-231` |
| `burst-bounds` indexes the file name for the frame ID before checking the sensor | `cli/download.py:276-281` |

**What is demonstrated.** A local HTTP server (a thread in this kernel, no network) serves a good
file, an HTML "error" page with status 200, a body shorter than its `Content-Length`, a slow
endpoint and a flaky one. The **old** module is taken from git (`git show d0901b3:...`) and
imported next to the **new** one, and both are run against each endpoint:

1. a good file;
2. an HTML error page: the old code saves it as a `.tenv8` and reuses it, the new code rejects it;
3. a truncated body;
4. a write interrupted half-way (disk full), and a leftover `.part`;
5. a slow server: the old code waits, the new code times out;
6. HTTP 500 and 429: the old code fails at once, the new code retries with backoff;
7. an empty TROPO search and a non-S1 file name at the CLI.

## Setup: a local server and the two versions of the module

In [2]:
import threading
import time
from collections import Counter
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

GOOD_BODY = b"".join(b"%.4f 1.0 2.0 3.0 0.1 0.1 0.1 0\n" % (2016 + i / 100) for i in range(400))
HTML_BODY = b"<!DOCTYPE html>\n<html><body><h1>Service temporarily unavailable</h1></body></html>\n"
HITS = Counter()          # requests seen per endpoint
SLOW_SECONDS = 3


class Handler(BaseHTTPRequestHandler):
    """Endpoint = first path segment: /<kind>/IGS20/000001_IGS20.tenv8."""

    def _send(self, body, content_type="text/plain", status=200, length=None):
        self.send_response(status)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(body) if length is None else length))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        kind = self.path.strip("/").split("/")[0]
        HITS[kind] += 1
        if kind == "good":
            self._send(GOOD_BODY)
        elif kind == "html":                      # status 200, but an error page
            self._send(HTML_BODY, "text/html; charset=utf-8")
        elif kind == "truncated":                 # announces the full length, sends half
            self._send(GOOD_BODY[: len(GOOD_BODY) // 2], length=len(GOOD_BODY))
        elif kind == "slow":                      # answers only after SLOW_SECONDS
            time.sleep(SLOW_SECONDS)
            self._send(GOOD_BODY)
        elif kind in ("flaky500", "flaky429"):    # fails twice, then works
            if HITS[kind] <= 2:
                self._send(b"try again later\n", status=int(kind[-3:]))
            else:
                self._send(GOOD_BODY)
        else:
            self.send_error(404)

    def log_message(self, *args):
        pass


class Server(ThreadingHTTPServer):
    def handle_error(self, request, client_address):
        pass                                      # a client that hung up on /slow


server = Server(("127.0.0.1", 0), Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
BASE = f"http://127.0.0.1:{server.server_address[1]}"
print("serving on", BASE, "| good file:", len(GOOD_BODY), "bytes | HTML page:", len(HTML_BODY), "bytes")

serving on http://127.0.0.1:46255 | good file: 14400 bytes | HTML page: 83 bytes


In [3]:
import importlib.util
import shutil
import subprocess
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")   # tqdm, no widgets needed

import pandas as pd
import requests
import urllib3

BEFORE = "d0901b3"        # the commit before this fix
WORK = SCRATCH / "13_download_robustness"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)


def load_old(path: str, name: str):
    """Import a module as it was at commit BEFORE (`git show BEFORE:path`)."""
    source = subprocess.run(
        ["git", "show", f"{BEFORE}:{path}"], cwd=REPO, capture_output=True, text=True, check=True
    ).stdout
    file = WORK / f"{name}.py"
    file.write_text(source)
    spec = importlib.util.spec_from_file_location(name, file)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


old = load_old("src/cal_disp/download/_stage_unr.py", "stage_unr_old")
from cal_disp.download import _stage_unr as new

def short(path) -> str:
    """Path relative to the repository or the scratch folder, for printing."""
    for root in (REPO, SCRATCH):
        if Path(path).is_relative_to(root):
            return str(Path(path).relative_to(root))
    return str(path)


print(f"old: commit {BEFORE}, src/cal_disp/download/_stage_unr.py ->", short(old.__file__))
print("new:", short(new.__file__))
print("requests", requests.__version__, "| urllib3", urllib3.__version__)

old: commit d0901b3, src/cal_disp/download/_stage_unr.py -> 13_download_robustness/stage_unr_old.py
new: src/cal_disp/download/_stage_unr.py
requests 2.34.2 | urllib3 2.8.0


In [4]:
def listing(folder: Path) -> str:
    files = sorted(folder.glob("*")) if folder.exists() else []
    return ", ".join(f"{f.name} ({f.stat().st_size} B)" for f in files) or "(nothing)"


def attempt(module, kind: str, label: str, session=None, **kwargs) -> dict:
    """Run module.download_grid_file(1, ...) against the /<kind> endpoint."""
    folder = WORK / label / kind
    saved = module.GRID_BASE_URLS["constant"]
    module.GRID_BASE_URLS["constant"] = f"{BASE}/{kind}"
    if session is None:
        session = module.create_session(retries=0)
    start = time.monotonic()
    try:
        path = module.download_grid_file(1, folder, session=session, **kwargs)
        result = f"returned {path.name}"
    except Exception as e:  # report, do not stop the notebook
        result = f"raised {type(e).__name__}: {str(e)[:110]}"
    finally:
        module.GRID_BASE_URLS["constant"] = saved
    return {"code": label, "endpoint": kind, "result": result,
            "seconds": round(time.monotonic() - start, 1), "files in output dir": listing(folder)}


def compare(kind: str, **kwargs) -> pd.DataFrame:
    rows = [attempt(old, kind, "old", **kwargs), attempt(new, kind, "new", **kwargs)]
    return pd.DataFrame(rows).set_index("code")


pd.set_option("display.max_colwidth", None)

## 1. A good file

Both versions download it; the new one leaves no `.part` behind.

In [5]:
display(compare("good"))
assert (WORK / "new" / "good" / "000001_IGS20_constant.tenv8").read_bytes() == GOOD_BODY

,endpoint,result,seconds,files in output dir
code,,,,
old,good,returned 000001_IGS20_constant.tenv8,0.0,000001_IGS20_constant.tenv8 (14400 B)
new,good,returned 000001_IGS20_constant.tenv8,0.0,000001_IGS20_constant.tenv8 (14400 B)


## 2. An HTML error page with status 200

Data servers and the proxies in front of them answer with an HTML page ("temporarily
unavailable", a login page, a maintenance notice) and status 200 more often than with a 5xx code,
so `raise_for_status()` does not catch it.

In [6]:
display(compare("html"))
saved = WORK / "old" / "html" / "000001_IGS20_constant.tenv8"
print("old code saved as .tenv8:", saved.read_bytes()[:60], "...")
assert not (WORK / "new" / "html").exists() or not list((WORK / "new" / "html").glob("*"))

,endpoint,result,seconds,files in output dir
code,,,,
old,html,returned 000001_IGS20_constant.tenv8,0.0,000001_IGS20_constant.tenv8 (83 B)
new,html,"raised DownloadError: http://127.0.0.1:46255/html/IGS20/000001_IGS20.tenv8 returned an HTML page (text/html; charset=utf-8), not a d",0.0,(nothing)


old code saved as .tenv8: b'<!DOCTYPE html>\n<html><body><h1>Service temporarily unavaila' ...


In [7]:
# The server "recovers": the same URL now serves the real file. Re-run both.
hits_before = HITS["good"]
for module, label in [(old, "old"), (new, "new")]:
    # point /good at the folder that was used for /html
    folder = WORK / label / "html"
    module.GRID_BASE_URLS["constant"], saved_url = f"{BASE}/good", module.GRID_BASE_URLS["constant"]
    path = module.download_grid_file(1, folder, session=module.create_session(retries=0))
    module.GRID_BASE_URLS["constant"] = saved_url
    body = path.read_bytes()
    print(f"{label}: {path.name}: {len(body)} bytes, starts with {body[:24]!r}  ->",
          "CORRUPT FILE REUSED" if body == HTML_BODY else "good file downloaded")
print("requests sent to the recovered server:", HITS["good"] - hits_before, "(only the new code asked again)")

try:
    pd.read_csv(WORK / "old" / "html" / "000001_IGS20_constant.tenv8", sep=r"\s+", header=None).astype(float)
except Exception as e:
    print(f"\nreading the old file as a time series fails later, far from the cause: {type(e).__name__}: {str(e)[:80]}")

old: 000001_IGS20_constant.tenv8: 83 bytes, starts with b'<!DOCTYPE html>\n<html><b'  -> CORRUPT FILE REUSED
new: 000001_IGS20_constant.tenv8: 14400 bytes, starts with b'2016.0000 1.0 2.0 3.0 0.'  -> good file downloaded
requests sent to the recovered server: 1 (only the new code asked again)

reading the old file as a time series fails later, far from the cause: ParserError: Error tokenizing data. C error: Expected 2 fields in line 2, saw 3



The old code wrote the HTML page to `000001_IGS20_constant.tenv8`, and because that file
now *exists* every later run returns it without asking the server again — the staging directory is
poisoned until somebody deletes the file by hand. The new code checks the `Content-Type` and the
first bytes of the body, raises `DownloadError`, and leaves nothing behind, so the next run
downloads the real file.

## 3. A truncated body

The server announces `Content-Length: <full size>` but closes the connection after half of it.

In [8]:
display(compare("truncated"))

,endpoint,result,seconds,files in output dir
code,,,,
old,truncated,"raised ChunkedEncodingError: ('Connection broken: IncompleteRead(7200 bytes read, 7200 more expected)', IncompleteRead(7200 bytes read, 720",0.0,(nothing)
new,truncated,"raised ChunkedEncodingError: ('Connection broken: IncompleteRead(7200 bytes read, 7200 more expected)', IncompleteRead(7200 bytes read, 720",0.0,(nothing)


With urllib3 2.x (this kernel) the HTTP client itself notices the short body and raises, in the old
code as well. urllib3 1.x did **not** enforce `Content-Length` and handed the short body to the
caller silently; `pyproject.toml` does not require urllib3 2. The next cell reproduces that client
behaviour with a stand-in session that returns half the body with the full `Content-Length`.

In [9]:
class ShortBodyResponse:
    """What a client that does not enforce Content-Length returns."""

    headers = {"Content-Type": "text/plain", "Content-Length": str(len(GOOD_BODY))}
    content = GOOD_BODY[: len(GOOD_BODY) // 2]

    def raise_for_status(self): pass
    def iter_content(self, chunk_size): yield self.content
    def __enter__(self): return self
    def __exit__(self, *exc): return False


class ShortBodySession:
    def get(self, url, **kwargs): return ShortBodyResponse()


rows = [attempt(old, "short-body", "old", session=ShortBodySession()),
        attempt(new, "short-body", "new", session=ShortBodySession())]
display(pd.DataFrame(rows).set_index("code"))
assert (WORK / "old" / "short-body" / "000001_IGS20_constant.tenv8").stat().st_size == len(GOOD_BODY) // 2
assert not list((WORK / "new" / "short-body").glob("*"))

,endpoint,result,seconds,files in output dir
code,,,,
old,short-body,returned 000001_IGS20_constant.tenv8,0.0,000001_IGS20_constant.tenv8 (7200 B)
new,short-body,raised DownloadError: http://127.0.0.1:46255/short-body/IGS20/000001_IGS20.tenv8: received 7200 bytes but Content-Length is 14400; d,0.0,(nothing)


## 4. A write interrupted half-way, and a leftover `.part`

`write_bytes` creates the final file first and then fills it. If the write stops half-way — disk
full on the scratch volume, the job killed — a partial file with the final name stays behind. The
cell simulates a disk that fills up after half of the data was written (`OSError: ENOSPC`).

In [10]:
import errno
from unittest.mock import patch


def write_half_then_fail(self, data):
    """Path.write_bytes on a disk that fills up half-way."""
    with open(self, "wb") as f:
        f.write(data[: len(data) // 2])
    raise OSError(errno.ENOSPC, "No space left on device")


class HalfThenFail:
    """File object for the new code's streaming write, same failure."""

    def __init__(self, file): self.file = file
    def write(self, data):
        self.file.write(data[: len(data) // 2]); self.file.flush()
        raise OSError(errno.ENOSPC, "No space left on device")
    def __enter__(self): return self
    def __exit__(self, *exc): self.file.close(); return False


with patch.object(Path, "write_bytes", write_half_then_fail):
    first_old = attempt(old, "good", "old-diskfull")
with patch.object(new, "open", lambda path, mode: HalfThenFail(open(path, mode)), create=True):
    first_new = attempt(new, "good", "new-diskfull")
print("First run, disk fills up during the write:")
display(pd.DataFrame([first_old, first_new]).set_index("code"))

print("Second run, disk space is available again:")
second = pd.DataFrame([attempt(old, "good", "old-diskfull"), attempt(new, "good", "new-diskfull")]).set_index("code")
second["file is complete"] = [
    (WORK / label / "good" / "000001_IGS20_constant.tenv8").read_bytes() == GOOD_BODY
    for label in second.index
]
display(second)
assert list(second["file is complete"]) == [False, True]

First run, disk fills up during the write:


,endpoint,result,seconds,files in output dir
code,,,,
old-diskfull,good,raised OSError: [Errno 28] No space left on device,0.0,000001_IGS20_constant.tenv8 (7200 B)
new-diskfull,good,raised OSError: [Errno 28] No space left on device,0.0,(nothing)


Second run, disk space is available again:


,endpoint,result,seconds,files in output dir,file is complete
code,,,,,
old-diskfull,good,returned 000001_IGS20_constant.tenv8,0.0,000001_IGS20_constant.tenv8 (7200 B),False
new-diskfull,good,returned 000001_IGS20_constant.tenv8,0.0,000001_IGS20_constant.tenv8 (14400 B),True


In [11]:
# A process killed hard (SIGKILL, node failure) cannot clean up: the new code then leaves
# "<name>.part", never a file with the final name. The next run discards it.
folder = WORK / "new-killed" / "good"
folder.mkdir(parents=True)
(folder / "000001_IGS20_constant.tenv8.part").write_bytes(GOOD_BODY[:1000])
print("left by a killed run :", listing(folder))
row = attempt(new, "good", "new-killed")
print("after the next run   :", row["files in output dir"], "|", row["result"])
assert (folder / "000001_IGS20_constant.tenv8").read_bytes() == GOOD_BODY

left by a killed run : 000001_IGS20_constant.tenv8.part (1000 B)
after the next run   : 000001_IGS20_constant.tenv8 (14400 B) | returned 000001_IGS20_constant.tenv8


The old code keeps the half-written file and returns it as complete on the second run. The new
code streams to `<name>.part` and renames it to the final name only after the body passed the
checks (`Path.replace`, atomic on POSIX), so a file with the final name is always complete: on a
failure the `.part` is removed, and a `.part` that survived a hard kill is discarded by the next run.

## 5. A server that does not answer

`/slow` answers only after 3 seconds. `requests` has no default timeout, so the old code waits as
long as the server takes — for a server that never answers, forever, and the PGE stage hangs with
it. The new code passes `timeout=(connect, read)` on every request.

In [12]:
import inspect

print("default timeout of the new code (connect, read) in seconds:", new.DEFAULT_TIMEOUT)
print("download_file signature:", inspect.signature(new.download_file))

old_row = attempt(old, "slow", "old")           # waits for the full SLOW_SECONDS

# download_grid_file uses the 120 s default; call download_file with a 1 s read timeout
# so that the demonstration stays short.
start = time.monotonic()
target = WORK / "new" / "slow" / "000001_IGS20_constant.tenv8"
target.parent.mkdir(parents=True, exist_ok=True)
try:
    new.download_file(f"{BASE}/slow/IGS20/000001_IGS20.tenv8", target,
                      session=new.create_session(retries=0), timeout=(5, 1.0))
    result = "returned"
except requests.RequestException as e:
    result = f"raised {type(e).__name__}: ...{str(e)[-60:]}"
new_row = {"code": "new (read timeout 1 s)", "endpoint": "slow", "result": result,
           "seconds": round(time.monotonic() - start, 1), "files in output dir": listing(target.parent)}
display(pd.DataFrame([old_row, new_row]).set_index("code"))
assert new_row["seconds"] < SLOW_SECONDS and "timed out" in result

default timeout of the new code (connect, read) in seconds: (10.0, 120.0)
download_file signature: (url: 'str', output_path: 'Path', session: 'requests.Session | None' = None, timeout: 'tuple[float, float]' = (10.0, 120.0)) -> 'Path'


,endpoint,result,seconds,files in output dir
code,,,,
old,slow,returned 000001_IGS20_constant.tenv8,3.0,000001_IGS20_constant.tenv8 (14400 B)
new (read timeout 1 s),slow,"raised ConnectionError: ...7.0.0.1', port=46255): Read timed out. (read timeout=1.0)""))",1.0,(nothing)


The exception is a `requests.ConnectionError` whose message says `Read timed out`: the retry
adapter reports an exhausted read timeout that way (without the adapter it is `requests.ReadTimeout`).
With the default of `(10, 120)` a stalled download fails after at most two minutes per attempt
instead of blocking the run.

## 6. HTTP 500 and 429

`/flaky500` and `/flaky429` fail twice and then serve the file. The old session retried only
502/503/504 (and mounted its retry adapter only for `https://`; it is mounted for `http://` here
too, so that the comparison is about the status list and not about the local server's scheme).

In [13]:
rows = []
for kind in ["flaky500", "flaky429"]:
    HITS[kind] = 0
    old_session = old.create_session(retries=5, backoff=0.1)
    old_session.mount("http://", old_session.get_adapter("https://"))
    rows.append({**attempt(old, kind, "old", session=old_session), "requests sent": HITS[kind]})
    HITS[kind] = 0
    rows.append({**attempt(new, kind, "new", session=new.create_session(retries=5, backoff=0.1)),
                 "requests sent": HITS[kind]})
display(pd.DataFrame(rows).set_index(["endpoint", "code"]))
print("statuses retried  old:", sorted(old.create_session().get_adapter("https://").max_retries.status_forcelist),
      " new:", sorted(new.create_session().get_adapter("https://").max_retries.status_forcelist))

result  \
endpoint code                                                                                                                                
flaky500 old   raised HTTPError: 500 Server Error: Internal Server Error for url: http://127.0.0.1:46255/flaky500/IGS20/000001_IGS20.tenv8   
         new                                                                                          returned 000001_IGS20_constant.tenv8   
flaky429 old       raised HTTPError: 429 Client Error: Too Many Requests for url: http://127.0.0.1:46255/flaky429/IGS20/000001_IGS20.tenv8   
         new                                                                                          returned 000001_IGS20_constant.tenv8   

               seconds                    files in output dir  requests sent  
endpoint code                                                                 
flaky500 old       0.0                              (nothing)              1  
         new       0.2  000001_IGS20_constant.tenv8 (14400 B)              3  
flaky429 old       0.0                              (nothing)              1  
         new       0.2  000001_IGS20_constant.tenv8 (14400 B)              3

statuses retried  old: [502, 503, 504]  new: [429, 500, 502, 503, 504]


## 7. TROPO search without results, and the `burst-bounds` file-name check

The TROPO staging asks ASF for the scenes nearest to each sensing time. Here the search is replaced
by one that finds nothing (as for a date outside the archive, or during an ASF outage), and the
download call by a recorder, so no network is used.

In [14]:
from datetime import datetime

import asf_search as asf

from cal_disp.download import _stage_tropo as new_tropo

old_tropo = load_old("src/cal_disp/download/_stage_tropo.py", "stage_tropo_old")
times = [datetime(2022, 1, 11, 0, 26, 51), datetime(2022, 7, 22, 0, 26, 57)]


def run_tropo(module, label):
    downloaded = []
    with patch.object(module, "find_nearest_scenes", return_value=asf.ASFSearchResults([])), \
         patch.object(asf.ASFSearchResults, "download",
                      lambda self, **kw: downloaded.append(len(self))):
        try:
            module.download_tropo(times, WORK / f"tropo-{label}")
            outcome = "returned normally"
        except Exception as e:
            outcome = f"raised {type(e).__name__}: {e}"
    return {"code": label, "result": outcome, "scenes per download() call": downloaded or "-"}


display(pd.DataFrame([run_tropo(old_tropo, "old"), run_tropo(new_tropo, "new")]).set_index("code"))

,result,scenes per download() call
code,,
old,returned normally,[0]
new,"raised DownloadError: No TROPO scenes found in collection C3717139408-ASF within +/-12 h of 2 of 2 sensing time(s): 2022-01-11 00:26:51, 2022-07-22 00:26:57",-


In [15]:
from click.testing import CliRunner

old_cli = load_old("src/cal_disp/cli/download.py", "cli_download_old")
from cal_disp.cli import download as new_cli

disp_file = WORK / "OPERA_L3_DISP-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20250101T000000Z.nc"
disp_file.touch()
not_s1 = WORK / "OPERA_L3_DISP-NI_IW.nc"      # another sensor, and a shorter name
not_s1.touch()


def cli(module, tropo_module, command, args):
    """Run a CLI command of `module` on top of `tropo_module`'s download_tropo.

    The CLI imports `cal_disp.download.download_tropo` when the command runs, so
    the old CLI is paired with the old library function by patching that name.
    """
    with patch.object(tropo_module, "find_nearest_scenes", return_value=asf.ASFSearchResults([])), \
         patch.object(new_tropo, "download_tropo", tropo_module.download_tropo), \
         patch.object(asf.ASFSearchResults, "download", lambda self, **kw: None):
        result = CliRunner().invoke(getattr(module, command), args)
    error = result.exception if result.exception and not isinstance(result.exception, SystemExit) else None
    return {"exit code": result.exit_code,
            "output": result.output.strip()[-150:],
            "unhandled exception": repr(error) if error else "-"}


rows = {}
for label, module, tropo_module in [("old", old_cli, old_tropo), ("new", new_cli, new_tropo)]:
    rows[("tropo, no scenes found", label)] = cli(module, tropo_module, "tropo", ["-i", str(disp_file), "-o", str(WORK / f"cli-tropo-{label}")])
    rows[("burst-bounds, DISP-NI file", label)] = cli(module, tropo_module, "burst_bounds", ["-i", str(not_s1), "-o", str(WORK / f"cli-bb-{label}")])
table = pd.DataFrame(rows).T.sort_index(ascending=[False, False])
display(table)
assert table.loc[("tropo, no scenes found", "old"), "exit code"] == 0      # reported success
assert table.loc[("tropo, no scenes found", "new"), "exit code"] != 0
assert "IndexError" in table.loc[("burst-bounds, DISP-NI file", "old"), "unhandled exception"]
assert "Only DISP-S1" in table.loc[("burst-bounds, DISP-NI file", "new"), "output"]

exit code  \
tropo, no scenes found     old         0   
                           new         1   
burst-bounds, DISP-NI file old         1   
                           new         1   

                                                                                                                                                                       output  \
tropo, no scenes found     old                                           Download complete: /u/aurora-r0/govorcin/tmp/cal_disp_notebooks/13_download_robustness/cli-tropo-old   
                           new  Error: No TROPO scenes found in collection C3717139408-ASF within +/-12 h of 2 of 2 sensing time(s): 2022-01-11 00:26:51, 2022-07-22 00:26:57   
burst-bounds, DISP-NI file old                                                                                                                                                  
                           new                                                                                                Error: Only DISP-S1 products supported, got: NI   

                                                  unhandled exception  
tropo, no scenes found     old                                      -  
                           new                                      -  
burst-bounds, DISP-NI file old  IndexError('list index out of range')  
                           new                                      -

With no scenes found the old `download_tropo` called `download()` on an empty result set and
returned, and the command line reported `Download complete` with exit code 0; the missing
troposphere files would only be noticed by a later stage. The new code raises `DownloadError`
naming the sensing times, and the CLI exits with code 1.

For a file of another sensor the old `burst-bounds` read the frame ID from the name before it
looked at the sensor, so the name above ended in an `IndexError` traceback instead of the intended
message. The new code checks the sensor first.

In [16]:
server.shutdown()
server.server_close()
shutil.rmtree(WORK, ignore_errors=True)
print("server stopped, scratch files removed")

server stopped, scratch files removed


## Summary

| Situation | Old behaviour | New behaviour |
|---|---|---|
| HTML error page with status 200 | saved as the data file, reused forever | `DownloadError`, nothing written |
| Body shorter than `Content-Length` | depends on the HTTP client: urllib3 2 raises, urllib3 1 saved the short file | raises in both cases, nothing written |
| Write interrupted (disk full, kill) | partial file with the final name, reused forever | only `<name>.part`, removed or discarded by the next run |
| Server does not answer | waits without limit | fails after the read timeout (default 120 s, connect 10 s) |
| HTTP 429 / 500 | fails on the first response | retried with backoff, like 502/503/504 |
| No TROPO scenes found | "Download complete", exit code 0 | `DownloadError` with the sensing times, exit code 1 |
| `burst-bounds` on a non-S1 name | `IndexError` traceback for short names | "Only DISP-S1 products supported" |

The same cases are covered by `tests/test_download_unr.py`, `tests/test_download_tropo.py` and
`tests/test_cli_download.py`, which also use a local server and no network.